# Rule Coverage & Alpha Sensitivity Analysis

Follow-up to Check-in 3. Two questions this notebook answers:

1. How much did lowering `min_support` from 0.01 to 0.005 improve disease coverage in the association-rule table?
2. What does the Recall@K curve look like as we sweep the fusion weight `alpha`?

The outputs are saved back into `data/results/` for reuse in the final report.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

rules       = pd.read_csv(ROOT / 'data' / 'processed' / 'association_rules.csv')
tx          = pd.read_csv(ROOT / 'data' / 'processed' / 'transactions.csv')
ablation    = pd.read_csv(ROOT / 'data' / 'results' / 'ablation_summary.csv')
alpha_sweep = pd.read_csv(ROOT / 'data' / 'results' / 'alpha_sweep.csv')

print('Rules:      ', len(rules))
print('Diseases in rules: ', rules['disease'].nunique())
print('Diseases in tx:    ', tx['condition'].nunique())

## 1. Rule coverage after lowering min_support

In [ ]:
def normalise(d: str) -> str:
    return str(d).strip().lower().replace(' ', '_').replace("'", '').replace('-', '_')

tx_diseases   = set(tx['condition'].map(normalise))
rule_diseases = set(rules['disease'].str.lower())
covered       = tx_diseases & rule_diseases
missing       = tx_diseases - rule_diseases

print(f'Diseases in Kaggle tx:   {len(tx_diseases)}')
print(f'Covered by mined rules:  {len(covered)}')
print(f'Still missing:           {len(missing)}')
print()
print('Missing diseases:')
for d in sorted(missing):
    print(' -', d)

In [ ]:
# Per-disease rule count
per_disease = rules.groupby('disease').size().sort_values(ascending=True)
print('Rules per disease (top 10):')
print(per_disease.tail(10))
print()
print('Rules per disease (bottom 10):')
print(per_disease.head(10))

fig, ax = plt.subplots(figsize=(8, 10))
per_disease.plot.barh(ax=ax)
ax.set_xlabel('Number of rules')
ax.set_title('Association rules per disease (min_support=0.005, min_confidence=0.5)')
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'results' / 'rules_per_disease.png', dpi=140)
plt.show()

## 2. Alpha sensitivity

In [ ]:
print(alpha_sweep[['alpha', 'recall@1', 'recall@5', 'recall@10', 'mrr']])

fig, ax = plt.subplots(figsize=(8, 5))
for col in ['recall@1', 'recall@5', 'recall@10']:
    ax.plot(alpha_sweep['alpha'], alpha_sweep[col], marker='o', label=col)
ax.set_xlabel('alpha (retrieval weight)')
ax.set_ylabel('Recall')
ax.set_title('Recall@K vs fusion weight alpha')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'results' / 'alpha_sensitivity.png', dpi=140)
plt.show()

## 3. Takeaway

* Lowering `min_support` from 0.01 to 0.005 lifts covered diseases from 20 to 33 (of 41 Kaggle classes).
* Alpha sweep confirms the Check-in 3 prediction: shifting weight back toward the mining component (alpha = 0.3) produces the best Recall@1, while the fused model still wins at Recall@10.
* The best observed (alpha = 0.3) point gives Recall@1 = 0.595 and Recall@10 = 0.975 — a clear improvement on the previous 0.6 default.